# 🧠 ترنم مهر — RAG شناساگر سال سوالات کنکور (Colab)

این نوت‌بوک **مدل پایه RAG** برای پروژه آموزشی ترنم مهر است.

ویژگی‌ها:
- بدون GPU، بدون HF token اجباری
- بارگذاری بانک سوالات از Hugging Face Space
- نرمال‌سازی فارسی + Retriever با TF-IDF (scikit-learn)
- شناسایی **سال و درس** سوال ورودی
- افزودن سوال جدید و ذخیره در بانک

نسخه‌ی آموزشی برای دیباگ، تست و گسترش بانک سوالات.

In [ ]:
# نصب سبک
!pip install -q requests pandas scikit-learn

In [ ]:
import re, json, requests
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# آدرس بانک سوالات روی Hugging Face Space
EXAMS_JSON_URL = 'https://sosa123454321-taranom-exam-rag.static.hf.space/data/exams.json'
print('Exams JSON:', EXAMS_JSON_URL)

## ۱) بارگذاری بانک سوالات

In [ ]:
resp = requests.get(EXAMS_JSON_URL, timeout=60)
resp.raise_for_status()
exams = resp.json()
print('تعداد سوالات:', len(exams))
df = pd.DataFrame(exams)
df[['year','subject','field','question']].head()

## ۲) نرمال‌سازی فارسی و ساخت متن RAG

In [ ]:
def normalize_fa(text):
    text = str(text or '')
    repl = {'ي':'ی','ك':'ک','ۀ':'ه','ة':'ه','أ':'ا','إ':'ا','ؤ':'و','\u200c':' ','\u200f':' ','\ufeff':' '}
    for a,b in repl.items():
        text = text.replace(a,b)
    text = re.sub(r'[۰-۹]', lambda m: str('۰۱۲۳۴۵۶۷۸۹'.index(m.group())), text)
    text = re.sub(r'[٠-٩]', lambda m: str('٠١٢٣٤٥٦٧٨٩'.index(m.group())), text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def record_text(r):
    return normalize_fa(' '.join([r.get('question',''), r.get('subject',''), r.get('field',''),
                                   r.get('source',''), ' '.join(r.get('options',[])), r.get('explanation','')]))

docs = [record_text(r) for r in exams]
print('متن نمونه:', docs[0][:120])

## ۳) ساخت Retriever با TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3))
tfidf = vectorizer.fit_transform(docs)
print('ماتریس TF-IDF:', tfidf.shape)

def find_year(query, top_k=3):
    q = vectorizer.transform([normalize_fa(query)])
    sims = cosine_similarity(q, tfidf)[0]
    idx = sims.argsort()[::-1][:top_k]
    results = []
    for i in idx:
        if sims[i] > 0.05:
            r = exams[i]
            results.append({'score': round(float(sims[i]),3), 'year': r['year'],
                            'subject': r['subject'], 'field': r['field'],
                            'source': r.get('source',''), 'question': r['question'][:80]})
    return results

## ۴) تست: این سوال مربوط به کدام سال است؟

In [ ]:
# یک سوال کنکور را امتحان کنید
queries = [
    'پمپ سدیم پتاسیم به ازای خروج هر ۳ یون سدیم چند یون پتاسیم وارد می‌کند؟',
    'حد تابع sinx/x هنگام x نزدیک صفر چقدر است؟',
    'در فتوسنتز واکنش‌های نوری کجا رخ می‌دهد؟',
]
for q in queries:
    print('\nسوال:', q)
    for hit in find_year(q, 1):
        print(f'  → {hit["source"]} | {hit["subject"]} | تطبیق {hit["score"]}')

## ۵) افزودن سوال جدید به بانک (و ذخیره)

In [ ]:
# سوال جدید را اینجا وارد کنید
new_q = {
    'id': 'q-new-1',
    'question': 'سوال نمونه‌ی جدید شما اینجا...',
    'options': ['گزینه۱','گزینه۲','گزینه۳','گزینه۴'],
    'answer': 'گزینه۱',
    'year': '1404', 'subject': 'زیست‌شناسی', 'field': 'تجربی',
    'source': 'کنکور ۱۴۰۴ تجربی', 'explanation': 'توضیح کوتاه'
}
exams.append(new_q)
print('تعداد جدید:', len(exams))
# ذخیره برای آپلود به Hugging Face:
# with open('exams.json','w',encoding='utf-8') as f: json.dump(exams,f,ensure_ascii=False,indent=2)
# سپس فایل را در Space (پوشه data/) آپلود کنید.

## ۶) ارزیابی دقت retriever (اختیاری)

In [ ]:
correct = 0; total = 0
for r in exams:
    res = find_year(r['question'], 1)
    if res:
        total += 1
        if res[0]['year'] == r['year']:
            correct += 1
print(f'دقت بازیابی سال: {correct}/{total} = {100*correct/max(total,1):.1f}%')